# Simulate template stimulus responses

Loads a template waveform from `data/template_waveforms.pkl` (generated by `26_05_26_generate_template_waveforms.ipynb`) and runs multi-trial simulations.

**Run the generate notebook first** if `data/template_waveforms.pkl` does not exist.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from designer_waveform.models import RandomEINetwork, load_config
from designer_waveform.optics import OpticsConfig, SigmoidPowerCurve

In [ ]:
# ── Waveform source ───────────────────────────────────────────────────────
# 'template'  : load from data/template_waveforms.pkl
#               Set TEMPLATE_NAME to one of: 'pulse_25ms', 'pulse_135ms', 'train_25hz_1s'
# 'optimised' : load opt_waveform from an optimisation result pkl
#               Set OPT_RESULT_PATH to the relevant file
WAVEFORM_SOURCE = 'template'
TEMPLATE_NAME   = 'pulse_25ms'
OPT_RESULT_PATH = Path('../results/optimised_waveforms/own_pipeline_passive/all/own_pipeline_passive_all_optimisation_result.pkl')

# ── Silence padding ───────────────────────────────────────────────────────
PRE_SILENCE_MS  = 20.0    # silence before waveform onset (ms)
POST_SILENCE_MS = 100.0   # silence after waveform offset (ms)

# ── PSTH bin size for analysis and display ───────────────────────────────
BIN_SIZE_MS = 10.0

# ── Multi-run settings ───────────────────────────────────────────────────
N_RUNS    = 20
SEED_BASE = 1000

VARY_INIT_V       = True
VARY_CONNECTIVITY = True
VARY_WEIGHTS      = True

# ── Network settings ─────────────────────────────────────────────────────
N_EXC     = 2000
N_INH     = 500
T_PRE_MS  = 200.0
T_POST_MS = 100.0

# ── Power mode ────────────────────────────────────────────────────────────
# USE_POWER_MODE = True  : waveform amplitude is source power in mW.
# RESCALE_AMPLITUDE      : True  → rescale waveform peak to AMPLITUDE_MW
#                          False → use waveform at its native amplitude
#                          (only meaningful when USE_POWER_MODE=True and
#                          waveform is already in mW, e.g. an optimised result)
USE_POWER_MODE    = True
RESCALE_AMPLITUDE = True
AMPLITUDE_MW      = 4.55   # target peak power (mW) — used only when RESCALE_AMPLITUDE=True

In [ ]:
if WAVEFORM_SOURCE == 'template':
    TEMPLATES_PATH = Path('../data/template_waveforms.pkl')
    with open(TEMPLATES_PATH, 'rb') as f:
        templates = pickle.load(f)
    if TEMPLATE_NAME not in templates:
        raise KeyError(f'TEMPLATE_NAME={TEMPLATE_NAME!r} not found. Available: {list(templates)}')
    _entry         = templates[TEMPLATE_NAME]
    _inner_wf      = _entry['waveform']
    _orig_stim_dur = float(_entry['stim_dur_ms'])
    description    = _entry['description']
    _label         = TEMPLATE_NAME

elif WAVEFORM_SOURCE == 'optimised':
    with open(OPT_RESULT_PATH, 'rb') as f:
        _opt = pickle.load(f)
    _inner_wf      = _opt['opt_waveform']
    _orig_stim_dur = float(_opt['stim_dur_ms'])
    description    = f"Optimised waveform ({_opt['source']} / {_opt['target_layer']})"
    _label         = f"optimised_{_opt['source']}_{_opt['target_layer']}"

else:
    raise ValueError(f"WAVEFORM_SOURCE={WAVEFORM_SOURCE!r} must be 'template' or 'optimised'")

print(f'Source         : {WAVEFORM_SOURCE}')
print(f'Description    : {description}')
print(f'Waveform       : {_inner_wf}')
print(f'Inner duration : {_orig_stim_dur:.0f} ms')
print(f'With padding   : {PRE_SILENCE_MS:.0f} + {_orig_stim_dur:.0f} + {POST_SILENCE_MS:.0f} = '
      f'{PRE_SILENCE_MS + _orig_stim_dur + POST_SILENCE_MS:.0f} ms total')

In [ ]:
# ── Extend stim window with silence buffers ───────────────────────────────
STIM_DUR_MS = PRE_SILENCE_MS + _orig_stim_dur + POST_SILENCE_MS

CONFIG_PATH = Path('..') / 'configs' / 'random_ei.json'
cfg = load_config(CONFIG_PATH)
cfg.N_exc       = N_EXC
cfg.N_inh       = N_INH
cfg.t_pre_ms    = T_PRE_MS
cfg.t_post_ms   = T_POST_MS
cfg.t_stim_ms   = STIM_DUR_MS
cfg.psth_bin_ms = BIN_SIZE_MS

if USE_POWER_MODE:
    _optics = OpticsConfig.from_file(Path('..') / 'data' / 'optics_params.json')
    _curve  = SigmoidPowerCurve(i_max_pA=1175, half_sat_mW_mm2=0.0015, hill_n=1.0)
    model   = RandomEINetwork(cfg, optics=_optics, power_curve=_curve,
                              normalization='max_expression')
    MAX_POWER_MW = _optics.area_mm2 * 0.1 / _optics.total_transmission
    _amp_label = 'mW'
    if RESCALE_AMPLITUDE:
        _peak  = max(_inner_wf(np.linspace(0, _orig_stim_dur, 1000)).max(), 1e-9)
        _scale = AMPLITUDE_MW / _peak
        _amp_tag = f'{AMPLITUDE_MW:.2f}mw'
        print(f'Power mode ON  — rescaled to {AMPLITUDE_MW:.2f} mW peak')
    else:
        _scale   = 1.0
        _amp_tag = 'native_mw'
        print(f'Power mode ON  — native amplitude (no rescaling)')
    print(f'               max deliverable: {MAX_POWER_MW:.2f} mW')
else:
    model        = RandomEINetwork(cfg)
    MAX_POWER_MW = None
    _scale       = 1.0
    _amp_label   = 'AU'
    _amp_tag     = 'au'
    print('Power mode OFF — amplitude in AU [0, 1]')

# ── Build padded waveform: silence → scaled stimulus → silence ────────────
_base_wf = _inner_wf   # original, picklable, for saving
_pre_s, _dur_s, _sc = PRE_SILENCE_MS, _orig_stim_dur, _scale

def waveform(t, _w=_base_wf, _pre=_pre_s, _dur=_dur_s, _s=_sc):
    t    = np.asarray(t, dtype=float)
    mask = (t >= _pre) & (t < _pre + _dur)
    out  = np.zeros_like(t)
    if mask.any():
        out[mask] = _w(t[mask] - _pre) * _s
    return out

_run_tag   = f'{_label}_{_amp_tag}'
OUTPUT_DIR = Path('../results') / 'template_responses' / _run_tag
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Stim window: {_orig_stim_dur:.0f} ms + {PRE_SILENCE_MS:.0f} ms pre '
      f'+ {POST_SILENCE_MS:.0f} ms post = {STIM_DUR_MS:.0f} ms total')
print(f'Model built.  Opsin mean: {model._stim_dist_pA.mean():.1f} pA, '
      f'frac zero: {(model._stim_dist_pA == 0).mean():.3f}')
print(f'Output dir : {OUTPUT_DIR}')

In [ ]:
# ── Background drive sweep ────────────────────────────────────────────────
# Runs the model with a zero-amplitude waveform at several I_bg_exc_pA
# values to find the setting that gives the target spontaneous firing rate.
# Adjust BG_VALUES_pA and TARGET_BASELINE_HZ, then re-run.
# Apply the suggested value by setting cfg.I_bg_exc_pA below and
# re-running this cell.
import time as _time

TARGET_BASELINE_HZ = 1.0
BG_VALUES_pA       = [50.0, 100.0, 150.0, 200.0, 250.0, 300.0]

_zero_wf    = lambda t: np.zeros_like(np.asarray(t, dtype=float))
_orig_bg    = float(cfg.I_bg_exc_pA)
_bg_results = []   # (bg_pA, mean_hz)

for _bg in BG_VALUES_pA:
    cfg.I_bg_exc_pA = float(_bg)
    _m_tmp = RandomEINetwork(cfg, **({
        'optics': _optics, 'power_curve': _curve, 'normalization': 'max_expression'
    } if USE_POWER_MODE else {}))
    _t0  = _time.time()
    _out = _m_tmp.run(_zero_wf)
    _hz  = float(_out['psth_exc'].mean() / (BIN_SIZE_MS / 1000.0))
    _bg_results.append((_bg, _hz))
    print(f'  I_bg_exc = {_bg:>6.1f} pA  →  {_hz:.2f} Hz  ({_time.time()-_t0:.1f} s)')

cfg.I_bg_exc_pA = _orig_bg   # restore until user applies

_bgs  = np.array([r[0] for r in _bg_results])
_rates = np.array([r[1] for r in _bg_results])

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(_bgs, _rates, 'o-', color='steelblue', lw=1.5)
ax.axhline(TARGET_BASELINE_HZ, color='k', lw=1, ls='--',
           label=f'target: {TARGET_BASELINE_HZ:.1f} Hz')
ax.axvline(_orig_bg, color='tomato', lw=0.8, ls=':',
           label=f'current cfg: {_orig_bg:.0f} pA')
ax.set_xlabel('I_bg_exc_pA')
ax.set_ylabel('Spontaneous rate (Hz)')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

if _rates.min() <= TARGET_BASELINE_HZ <= _rates.max():
    _order     = np.argsort(_rates)
    _suggested = float(np.interp(TARGET_BASELINE_HZ, _rates[_order], _bgs[_order]))
    print(f'\n→ Suggested I_bg_exc_pA: {_suggested:.1f} pA')
    print('  Apply by setting cfg.I_bg_exc_pA below and re-running this cell:')
    print(f'  cfg.I_bg_exc_pA = {_suggested:.1f}')
    # ── Apply ──────────────────────────────────────────────────────────
    cfg.I_bg_exc_pA = _suggested
    model = RandomEINetwork(cfg, **({
        'optics': _optics, 'power_curve': _curve, 'normalization': 'max_expression'
    } if USE_POWER_MODE else {}))
    print(f'  Applied. Model rebuilt.')
elif _rates.max() < TARGET_BASELINE_HZ:
    print(f'\n→ All values below target. Extend BG_VALUES_pA above {BG_VALUES_pA[-1]:.0f} pA.')
else:
    print(f'\n→ All values above target. Extend BG_VALUES_pA below {BG_VALUES_pA[0]:.0f} pA.')

## Waveform preview

In [ ]:
_t_plot  = np.linspace(0, STIM_DUR_MS, int(STIM_DUR_MS / 0.1))
_wf_vals = waveform(_t_plot)
_ylim    = (-0.05 * _wf_vals.max(), _wf_vals.max() * 1.15)

fig, ax = plt.subplots(figsize=(max(7, STIM_DUR_MS / 80), 2.5))
ax.fill_between(_t_plot, 0, _wf_vals, alpha=0.35, color='steelblue')
ax.plot(_t_plot, _wf_vals, color='steelblue', lw=1.5)
ax.set_xlim(0, STIM_DUR_MS)
ax.set_ylim(*_ylim)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel(f'Source power ({_amp_label})' if USE_POWER_MODE else 'Envelope amplitude (AU)')
ax.set_title(f'{TEMPLATE_NAME} — {description}')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'waveform.png', dpi=150)
plt.show()

## Multi-run simulation

In [ ]:
_vary_str = ', '.join(
    s for s, v in [('init_v', VARY_INIT_V), ('connectivity', VARY_CONNECTIVITY),
                   ('weights', VARY_WEIGHTS)] if v
) or 'none'
print(f'Running {N_RUNS} simulations (varying: {_vary_str})...')

_all_psth  = []
_all_fine  = []
_spike_data_one = None   # store one run for raster

_fine_bin_ms   = max(1.0, BIN_SIZE_MS / 2.0)   # finer bin for display
_fine_edges    = np.arange(0, STIM_DUR_MS + _fine_bin_ms, _fine_bin_ms)
_fine_t        = 0.5 * (_fine_edges[:-1] + _fine_edges[1:])

for _i in range(N_RUNS):
    _r = model.run(waveform, seed=SEED_BASE + _i,
                   vary_init_v=VARY_INIT_V,
                   vary_connectivity=VARY_CONNECTIVITY,
                   vary_weights=VARY_WEIGHTS)
    _all_psth.append(_r['psth_exc'] / (BIN_SIZE_MS / 1000.0))

    _exc_m  = _r['spike_indices'] < cfg.N_exc
    _et     = _r['spike_times_ms'][_exc_m] - cfg.t_pre_ms
    _win    = (_et >= 0) & (_et <= STIM_DUR_MS)
    _cnt, _ = np.histogram(_et[_win], bins=_fine_edges)
    _all_fine.append(_cnt / cfg.N_exc / (_fine_bin_ms / 1000.0))

    if _i == 0:
        _spike_data_one = {
            'times':   _r['spike_times_ms'][_exc_m][_win] - 0,
            'indices': _r['spike_indices'][_exc_m][_win],
            'offset':  cfg.t_pre_ms,
        }
    if (_i + 1) % 5 == 0:
        print(f'  {_i + 1}/{N_RUNS}')

_psth_arr  = np.stack(_all_psth)
_fine_arr  = np.stack(_all_fine)
t_psth_ms  = model.run(waveform)['t_psth_ms']

mean_hz    = _psth_arr.mean(0)
sem_hz     = _psth_arr.std(0) / np.sqrt(N_RUNS)
mean_fine  = _fine_arr.mean(0)
sem_fine   = _fine_arr.std(0) / np.sqrt(N_RUNS)

print('Done.')

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, STIM_DUR_MS / 30), 4))

for _run_hz in _fine_arr:
    ax.plot(_fine_t, _run_hz, color='steelblue', lw=0.7, alpha=0.2)
ax.fill_between(_fine_t, mean_fine - sem_fine, mean_fine + sem_fine,
                color='steelblue', alpha=0.35)
ax.plot(_fine_t, mean_fine, color='steelblue', lw=2,
        label=f'Mean ± SEM (n={N_RUNS})')
ax.set_xlim(0, STIM_DUR_MS)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'{TEMPLATE_NAME} | {_fine_bin_ms:.0f} ms bins  [varying: {_vary_str}]')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'psth.png', dpi=150)
plt.show()

In [ ]:
N_RASTER = 200   # neurons to show

_sd  = _spike_data_one
_st  = _sd['times'] - _sd['offset']
_si  = _sd['indices']
_win = (_st >= 0) & (_st <= STIM_DUR_MS)
_rm  = (_si < N_RASTER) & _win

fig, axes = plt.subplots(2, 1, figsize=(max(9, STIM_DUR_MS / 30), 6),
                          gridspec_kw={'height_ratios': [1, 2]})

ax = axes[0]
ax.fill_between(_t_plot, 0, _wf_vals, alpha=0.3, color='steelblue')
ax.plot(_t_plot, _wf_vals, color='steelblue', lw=1.2)
ax.set_xlim(0, STIM_DUR_MS)
ax.set_ylabel(f'Power ({_amp_label})' if USE_POWER_MODE else 'Envelope (AU)')
ax.set_title(f'Raster — {N_RASTER} exc neurons, run 0  [{TEMPLATE_NAME}]')
ax.tick_params(bottom=False, labelbottom=False)
ax.spines[['top', 'right', 'bottom']].set_visible(False)

ax = axes[1]
ax.scatter(_st[_rm], _si[_rm], s=6, color='k', alpha=0.6, linewidths=0)
ax.set_xlim(0, STIM_DUR_MS)
ax.set_ylim(0, N_RASTER)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Neuron index')
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'raster.png', dpi=150)
plt.show()

In [ ]:
_save = {
    'template_name':    TEMPLATE_NAME if WAVEFORM_SOURCE == 'template' else None,
    'waveform_source':  WAVEFORM_SOURCE,
    'description':      description,
    'orig_stim_dur_ms': _orig_stim_dur,
    'pre_silence_ms':   PRE_SILENCE_MS,
    'post_silence_ms':  POST_SILENCE_MS,
    'stim_dur_ms':      STIM_DUR_MS,
    'bin_size_ms':      BIN_SIZE_MS,
    'fine_bin_ms':      _fine_bin_ms,
    'n_runs':           N_RUNS,
    'seed_base':        SEED_BASE,
    'use_power_mode':   USE_POWER_MODE,
    'rescale_amplitude': RESCALE_AMPLITUDE,
    'amplitude_mw':     AMPLITUDE_MW if (USE_POWER_MODE and RESCALE_AMPLITUDE) else None,
    'waveform_scale':   _scale,
    # original (un-padded, un-scaled) waveform — picklable
    # reconstruct full waveform with:
    #   out[t >= pre & t < pre+dur] = waveform(t[...] - pre) * waveform_scale
    'waveform':         _base_wf,
    't_psth_ms':        t_psth_ms,
    'runs_hz':          _psth_arr,
    'mean_hz':          mean_hz,
    'sem_hz':           sem_hz,
    'fine_t_ms':        _fine_t,
    'fine_mean_hz':     mean_fine,
    'fine_sem_hz':      sem_fine,
}
_save_path = OUTPUT_DIR / 'results.pkl'
with open(_save_path, 'wb') as f:
    pickle.dump(_save, f)
print(f'Saved to {_save_path}')